In [7]:
# %%
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb
import optuna
import joblib 

# Вимикаємо спам у консолі від Optuna (щоб бачити тільки прогрес-бар)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# %%
categories = ["Season", "Time", "Weather"]
models = {} 
label_encoders = {} # Зберігаємо енкодери для кожної категорії

numeric_features = ['BPM', 'Energy', 'Dance', 'Acoustic', 'Valence']
categorical_features = ['Key'] 

for cat in categories:
    print(f"\n🚀 Оптимізуємо та тренуємо XGBoost для: {cat}")
    print("-" * 50)
    
    # Завантажуємо датасет
    df = pd.read_csv(f"{cat.lower()}.csv")
    df = df.dropna(subset=numeric_features + categorical_features + [f'{cat}_Label'])
    
    X = df[numeric_features + categorical_features]
    y_text = df[f'{cat}_Label']
    
    # ❗️ XGBoost вимагає числові лейбли, тому перетворюємо текст на цифри
    le = LabelEncoder()
    y = le.fit_transform(y_text)
    label_encoders[cat] = le
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Функція для Optuna
    def objective(trial):
        # 🔥 Купа параметрів для жорсткого тюнінгу
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0.0, 5.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 10.0, log=True),
        }
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), numeric_features),
                ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
            ])
        
        clf = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', xgb.XGBClassifier(**param, random_state=42, eval_metric='mlogloss'))
        ])
        
        clf.fit(X_train, y_train)
        return clf.score(X_test, y_test)

    print("⏳ Запуск Optuna (50 спроб)...")
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)
    
    print(f"✨ Найкраща точність при пошуку: {study.best_value:.4f}")
    
    # Тренуємо фінальну модель з найкращими знайденими параметрами
    best_params = study.best_params
    
    final_preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])
    
    final_clf = Pipeline(steps=[
        ('preprocessor', final_preprocessor),
        ('classifier', xgb.XGBClassifier(**best_params, random_state=42, eval_metric='mlogloss'))
    ])
    
    final_clf.fit(X_train, y_train)
    
    # Оцінюємо фінальну модель
    score = final_clf.score(X_test, y_test)
    print(f"📊 Фінальна точність на тесті ({cat}): {score:.4f}")
    
    models[cat] = final_clf

# %%
# Завантажуємо тестовий плейлист summer_test.csv
test_df = pd.read_csv('summer_test.csv')

features = ['BPM', 'Energy', 'Dance', 'Acoustic', 'Valence', 'Key']
numeric_features = ['BPM', 'Energy', 'Dance', 'Acoustic', 'Valence']

print(f"\nСтворюємо портрет плейлиста з {len(test_df)} треків")
print("=" * 50)

# Збираємо портрет у словник, щоб безпечно перетворити в DataFrame
playlist_dict = test_df[numeric_features].mean().to_dict()
playlist_dict['Key'] = test_df['Key'].mode()[0]

print("Середні значення плейлиста:")
for feat, val in playlist_dict.items():
    if isinstance(val, float):
        print(f"  - {feat}: {val:.2f}")
    else:
        print(f"  - {feat}: {val}")

print("\n" + "=" * 50)
print("Прогнози XGBoost для портрета плейлиста:")
print("=" * 50)

# Перетворюємо на DataFrame в 1 рядок
input_df = pd.DataFrame([playlist_dict])

for cat in categories:
    model = models[cat]
    le = label_encoders[cat]
    
    # Отримуємо ймовірності
    probabilities = model.predict_proba(input_df)[0]
    
    # Розшифровуємо класи (0, 1, 2...) назад у слова
    class_indices = model.classes_
    class_names = le.inverse_transform(class_indices)
    
    print(f"\n[{cat.upper()}] Профіль:")
    
    # Сортуємо результати від найбільшого до найменшого
    results = list(zip(class_names, probabilities))
    results.sort(key=lambda x: x[1], reverse=True)
    
    for label, prob in results:
        print(f"  - {label}: {prob * 100:.1f}%")


🚀 Оптимізуємо та тренуємо XGBoost для: Season
--------------------------------------------------
⏳ Запуск Optuna (50 спроб)...
✨ Найкраща точність при пошуку: 0.5027
📊 Фінальна точність на тесті (Season): 0.5027

🚀 Оптимізуємо та тренуємо XGBoost для: Time
--------------------------------------------------
⏳ Запуск Optuna (50 спроб)...
✨ Найкраща точність при пошуку: 0.4524
📊 Фінальна точність на тесті (Time): 0.4524

🚀 Оптимізуємо та тренуємо XGBoost для: Weather
--------------------------------------------------
⏳ Запуск Optuna (50 спроб)...
✨ Найкраща точність при пошуку: 0.4735
📊 Фінальна точність на тесті (Weather): 0.4735

Створюємо портрет плейлиста з 53 треків
Середні значення плейлиста:
  - BPM: 122.94
  - Energy: 68.25
  - Dance: 69.42
  - Acoustic: 12.96
  - Valence: 51.11
  - Key: A Major

Прогнози XGBoost для портрета плейлиста:

[SEASON] Профіль:
  - summer: 45.9%
  - winter: 20.9%
  - spring: 19.8%
  - autumn: 13.3%

[TIME] Профіль:
  - morning: 47.6%
  - evening: 22.6%